# DepthWizard v5 — Kaggle 2× T4 train on GAMUS + SynRS3D only (no DFC23)

**Accelerator `GPU T4 x2` · Internet ON · Add-ons → Secrets: `HF_TOKEN` and `GITHUB_PAT`.**
Run with **Save Version → Save & Run All (Commit)**; an interactive session idles out.

**Attach as inputs** (all `abhaydkale232/…`): `depthwizard-gamus` (train/val/test),
`depthwizard-synrs3d-g05` and `depthwizard-synrs3d-g1`. That's all. **Don't attach**
`dfc23-track2-height-estimation`. **Don't attach** `depthwizard-india-labeled` either: it
is DFC23's 131 New Delhi scenes, so it counts as DFC data too. v4's `best.pt` is pulled from
the private Hub repo `akashch1512/SingleViewHeightEstimator` (`best_3.7/best.pt`).

The profile is the same as `kaggle_test.ipynb` (`V4_modal/v5_flags.py`, T4-sized). Only the
data changes: `--datasets gamus,synrs3d_g05,synrs3d_g1` with v5's own sampler weights for those
three sources. The coarse-label path (`coarse_label_sources`) is off, since both of its sources
are DFC23. Everything else in v5 still runs: the detail branch, the Cartosat pansharp/gray augs
and the v3 reverts. Validation and `best.pt` selection use `gamus/val`. Cell 3 scores v4 and
this run on the same held-out `gamus/test` tiles.

`WARM_START = True` starts from v4's weights, and `False` trains from scratch.
First run of this notebook on a fresh setup: commit once with `SMOKE = True` (~15 min).

In [ ]:
# ═══ 1. Code, data ═══════════════════════════════════════════════════════
import os, sys, glob, shlex, subprocess, pathlib

WARM_START = True     # True: v5 starts from v4's weights.  False: from scratch.
V4_CKPT   = ""        # a Kaggle path to v4's best.pt; "" -> downloaded from V4_HF
V4_HF     = ("akashch1512/SingleViewHeightEstimator", "best_3.7/best.pt")  # private repo
SMOKE     = False     # True for the first commit
RESUME    = ""        # a dead session: "/kaggle/input/<prev>/outputs/v5_gs_<tag>/last_full.pt"
MAX_MINUTES = 540     # training budget; the whole notebook is ~10.5 h of the 12 h cap
EPOCHS    = 14
SLIDING_TILES = 0     # sliding-window test tiles in cell 3; each costs ~minutes on T4
RUN_TESTS = True
# The only training sources.  gamus first: best.pt is selected on the first val source.
SOURCES   = {"gamus": 3, "synrs3d_g05": 1, "synrs3d_g1": 0.5}   # v5_flags' weights

REPO = "/kaggle/temp/DepthWizard"     # not /kaggle/working: the clone holds the PAT
CODE = f"{REPO}/FineTunning/v5"
DATA = "/kaggle/temp/dwdata"          # symlinks to the attached stores
TAG  = "v4init" if WARM_START else "scratch"
OUT  = f"/kaggle/working/outputs/v5_gs_{TAG}" + ("_smoke" if SMOKE else "")
GIT_URL = "github.com/AbhayKale332/DepthWizard.git"
sh = lambda c: subprocess.run(c, shell=True, check=True)

if not os.path.isdir(REPO):
    pat = ""
    try:
        from kaggle_secrets import UserSecretsClient
        pat = UserSecretsClient().get_secret("GITHUB_PAT")
    except Exception as e:
        print("no GITHUB_PAT secret (", e, ") - trying an anonymous clone")
    url = f"https://x-access-token:{pat}@{GIT_URL}" if pat else f"https://{GIT_URL}"
    r = subprocess.run(["git", "clone", "--depth", "1", url, REPO], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit("git clone failed: " + (r.stderr.replace(pat, "***") if pat else r.stderr))
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", f"https://{GIT_URL}"], check=True)
os.chdir(CODE); sys.path.insert(0, CODE)
sh("pip install -q -r requirements.txt")

if RUN_TESTS:
    r = subprocess.run("python -m pytest -q --tb=short -rf", shell=True, capture_output=True, text=True)
    print(((r.stdout or "") + (r.stderr or ""))[-6000:])
    if r.returncode:
        raise SystemExit(f"pytest exited {r.returncode} — output above. RUN_TESTS = False skips.")

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # DINOv3-SAT is gated

if not V4_CKPT and V4_HF:         # 2.5 GB; the baseline in cell 3 either way
    from huggingface_hub import hf_hub_download
    V4_CKPT = hf_hub_download(*V4_HF, token=os.environ["HF_TOKEN"], local_dir="/kaggle/temp/ckpt")
if V4_CKPT and not os.path.isfile(V4_CKPT):
    raise SystemExit(f"V4_CKPT not found: {V4_CKPT}")
print(f"v4 checkpoint: {V4_CKPT or 'none'}   init: {'v4' if WARM_START else 'scratch'}")

# the attached packed stores -> one data_root of symlinks (nothing is copied, nothing packed)
sh(f"DW_DATA_ROOT={DATA} bash run_kaggle.sh link")
need = [f"{s}/train" for s in SOURCES] + ["gamus/val", "gamus/test"]
missing = [p for p in need if not os.path.isfile(f"{DATA}/{p}/index.json")]
if missing:
    raise SystemExit(f"not attached / not linked: {missing}")
extra = sorted(set(os.listdir(DATA)) - set(SOURCES) - {".stores"})
if extra:
    print(f"note: also attached but NOT trained on: {extra}")
sh(f"du -shL {DATA}/*; df -h /kaggle/temp | tail -1")

In [ ]:
# ═══ 2. Train: the v5 profile, T4-sized, GAMUS + SynRS3D only ═════════════
sys.path.insert(0, f"{REPO}/FineTunning/V4_modal")
from v5_flags import FLAGS as V5

init = RESUME or (V4_CKPT if WARM_START else "")
flags = {**V5,
    "data_root": DATA, "output_dir": OUT, "resume": init,
    # no DFC23, no india_labeled (= DFC23's New Delhi scenes)
    "datasets": ",".join(SOURCES),
    "sampler_weights": ",".join(f"{k}:{v}" for k, v in SOURCES.items()),
    # both coarse sources are gone; "" is dropped below, so config.py's "" default applies
    "coarse_label_sources": "",
    # Turing: no bf16 -> fp16 + GradScaler; encoder checkpointing to fit 15 GB
    "amp_dtype": "fp16", "grad_checkpoint_encoder": "true",
    # 2 ranks x 1 x 16 = 32, the v5 profile's effective batch (16 x 2).  Batch 2 OOMed
    # at the e3 unfreeze (14.5 of 14.6 GiB), and fp16 now runs the decoder trunk in fp32.
    # Eval batch 16 OOMed at the first eval (fp32 trunk + upsampler at H/2); 4 leaves
    # room for the unfrozen blocks' grads and Adam state, which stay resident during eval.
    "batch_size": "1", "grad_accum": "16", "eval_batch_mult": "4",
    "num_workers": "2", "prefetch_factor": "2",
    # all 24 blocks at llrd 0.80 halves T4 throughput (4.9 -> 2.3 img/s); keep v4-T4's 16 / 0.90
    "encoder_unfreeze_blocks": "16", "llrd": "0.90",
    "epochs": str(EPOCHS), "max_minutes": str(MAX_MINUTES), "session_minutes": "0",
    # held-out / sliding / TTA / ONNX at the end of train.py is hours on T4; cell 3 scores instead
    "test_sources": "", "final_sliding_eval": "false", "tta": "false", "export_onnx": "false",
}
if WARM_START:
    # v4's Head B range: v5_flags' 150 would stretch the loaded bin widths 1.25x.
    # Not a weight, so a resume must pass it again.
    flags["bin_max_m"] = "120"
args = " ".join(f"--{k} {shlex.quote(str(v))}" for k, v in flags.items() if str(v) != "")
if SMOKE:
    args += " --smoke"
print("python train.py", args)

# torch 2.10 reads PYTORCH_ALLOC_CONF; the CUDA-named one is the old spelling
os.environ.update(PYTORCH_ALLOC_CONF="expandable_segments:True",
                  PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", NCCL_P2P_DISABLE="1",
                  OMP_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false")
sh(f"torchrun --standalone --nnodes=1 --nproc_per_node=2 train.py {args}")

In [ ]:
# ═══ 3. v4 vs v5 on the same held-out tiles ═══════════════════════════════
# Centre crop over all gamus/test tiles, plain (no TTA), plus 12 qualitative strips
# with shadow IoU — the same tiles for both (eval_test.py seeds them).
if not SMOKE:
    common = (f"--data_root {DATA} --source gamus --split test --tiles 0 --skip_tta "
              f"--sliding_tiles {SLIDING_TILES} --qualitative 12 --max_valid_height_m 150 "
              "--batch_size 8 --eval_batch_mult 2 --num_workers 4")
    # Each checkpoint in the precision it trained in.  v4 trained in bf16 on the H100;
    # scored in fp16 it came back at RMSE 10.9 with a +22 m bias on 20m+, against ~3.7
    # on its own val, so something in it overflows fp16's 65504.  Turing has no bf16
    # units, so torch emulates it: slower than fp16, but with fp32's range.
    runs = [("v4", V4_CKPT, "bf16")] if V4_CKPT else []
    runs.append((f"v5_{TAG}", f"{OUT}/best.pt", "fp16"))
    for tag, ck, dt in runs:
        sh(f"python eval_test.py --ckpt {ck} {common} --amp_dtype {dt} --out {OUT}/test_{tag}")
    dirs = ",".join(f"{OUT}/test_{t}" for t, *_ in runs)
    labels = ",".join(t for t, *_ in runs)
    sh(f"python eval_test.py --compare {dirs} --labels {labels} --md {OUT}/v4_vs_v5_{TAG}.md")

## If the session dies

Attach this version's output as an input to a fresh copy of the notebook. Set
`RESUME = "/kaggle/input/<name>/outputs/v5_gs_<v4init|scratch>/last_full.pt"`, leave
`WARM_START` as it was, and commit again. `/kaggle/temp` gets wiped, so the notebook
re-clones and re-links the stores first. Nothing needs re-packing because there's no DFC23.

Reading the result: compare `test_v5_*` with `test_v4` in `v4_vs_v5_<tag>.md`. v4 trained
with DFC23 and india_labeled, so this run shows how much v5 loses or gains without them
on GAMUS test.